# KPI Data Checks for Retail Sales ELT
Notebook ini memverifikasi kondisi data yang mendukung KPI dashboard: Total Sales, Total Profit, Profit Margin, Monthly Sales Trend, Top Products, dan Sales per Region.

## 1. Setup dan koneksi

In [9]:
import os
import pandas as pd
import clickhouse_connect
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
RAW_DIR = os.getenv('RAW_DIR', 'data')
CH_HOST = os.getenv('CH_HOST', 'localhost')
CH_PORT = int(os.getenv('CH_PORT', 8123))
CH_USER = os.getenv('CH_USER', 'admin')
CH_PASSWORD = os.getenv('CH_PASSWORD', 'admin123')

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
)
print('✅ Connected to ClickHouse')

✅ Connected to ClickHouse


## 2. Bronze ingestion validation

In [10]:
csv_path = os.path.join('..', RAW_DIR, 'samplesuperstore.csv')
df_csv = pd.read_csv(csv_path, encoding='utf-8')
csv_count = len(df_csv)

table_name = 'bronze.bronze_orders'
result = client.query(f'SELECT count() FROM {table_name}')
ch_count = result.result_rows[0][0]
print(f'CSV rows          : {csv_count:,}')
print(f'Bronze rows       : {ch_count:,}')
print('✅ Bronze ingestion row count check' if ch_count == csv_count else '⚠️ Bronze ingestion mismatch')

CSV rows          : 10,194
Bronze rows       : 10,194
✅ Bronze ingestion row count check


## 3. Silver model sanity checks

In [11]:
silver_table = 'silver.silver_orders'
df_silver = client.query_df(f'SELECT * FROM {silver_table} LIMIT 5')
print('Silver preview:')
display(df_silver)

schema_silver = client.query_df(
    f"SELECT name, type FROM system.columns WHERE database='silver' AND table='silver_orders' ORDER BY position"
)
print('Silver schema:')
display(schema_silver)

Silver preview:


,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,state_province,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,US-2023-103800,2023-03-01,2023-07-01,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,US-2023-112326,2023-04-01,2023-08-01,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,US-2023-112326,2023-04-01,2023-08-01,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,US-2023-112326,2023-04-01,2023-08-01,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,US-2023-141817,2023-05-01,2023-12-01,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


Silver schema:


,name,type
0,order_id,String
1,order_date,DateTime
2,ship_date,DateTime
3,ship_mode,String
4,customer_id,String
5,customer_name,String
6,segment,String
7,country_region,String
8,city,String
9,state_province,String


## 4. KPI checks for dashboard metrics

In [12]:
print('### KPI Card values')
kpi_query = '''
SELECT
    sum(sales) AS total_sales,
    sum(profit) AS total_profit,
    round(100.0 * sum(profit) / NULLIF(sum(sales), 0), 2) AS profit_margin
FROM silver.silver_orders
'''
df_kpi = client.query_df(kpi_query)
display(df_kpi)

### KPI Card values


,total_sales,total_profit,profit_margin
0,2.326534e+06,292296.8146,12.56


### Monthly Sales Trend

In [13]:
trend_query = '''
SELECT
    toStartOfMonth(order_date) AS order_month,
    sum(sales) AS total_sales,
    sum(profit) AS total_profit,
    round(100.0 * sum(profit) / NULLIF(sum(sales), 0), 2) AS profit_margin
FROM silver.silver_orders
GROUP BY order_month
ORDER BY order_month
'''
df_trend = client.query_df(trend_query)
display(df_trend)

,order_month,total_sales,total_profit,profit_margin
0,2023-01-01,29234.8660,4638.6546,15.87
1,2023-02-01,12857.8920,2611.2185,20.31
2,2023-03-01,56044.8060,287.4190,0.51
3,2023-04-01,24751.9960,4616.3514,18.65
4,2023-05-01,31973.0740,4140.0683,12.95
5,2023-06-01,29361.3826,4522.9796,15.40
6,2023-07-01,36432.5960,-1567.7287,-4.30
7,2023-08-01,38863.4175,2528.3876,6.51
8,2023-09-01,66988.4218,10494.0734,15.67
9,2023-10-01,34783.6630,4113.8375,11.83


### Top Products by Profit and Sales
Top 5 produk akan terlihat dari tabel ringkasan produk berikut.

In [14]:
product_query = '''
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    total_sales,
    total_profit,
    profit_margin,
    total_quantity
FROM gold.product_performance
ORDER BY total_profit DESC
LIMIT 5
'''
df_top_profit = client.query_df(product_query)
print('Top 5 products by profit:')
display(df_top_profit)

product_query_sales = '''
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    total_sales,
    total_profit,
    profit_margin,
    total_quantity
FROM gold.product_performance
ORDER BY total_sales DESC
LIMIT 5
'''
df_top_sales = client.query_df(product_query_sales)
print('Top 5 products by sales:')
display(df_top_sales)

Top 5 products by profit:


,product_id,product_name,category,sub_category,total_sales,total_profit,profit_margin,total_quantity
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,61599.824,25199.9280,40.91,20
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,27453.384,7753.0390,28.24,31
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,18839.686,6983.8836,37.07,38
3,TEC-CO-10003763,Canon PC1060 Personal Laser Copier,Technology,Copiers,11619.834,4570.9347,39.34,19
4,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,Technology,Machines,18374.895,4094.9766,22.29,12


Top 5 products by sales:


,product_id,product_name,category,sub_category,total_sales,total_profit,profit_margin,total_quantity
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,61599.824,2.519993e+04,40.91,20
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,27453.384,7.753039e+03,28.24,31
2,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,22638.480,-1.811078e+03,-8.00,6
3,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,21870.576,1.136868e-13,0.00,39
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,19823.479,2.233505e+03,11.27,37


### Sales per Region

In [15]:
region_query = '''
SELECT
    region,
    total_sales,
    total_profit,
    profit_margin,
    total_quantity
FROM gold.region_sales
ORDER BY total_sales DESC
'''
df_region = client.query_df(region_query)
display(df_region)

,region,total_sales,total_profit,profit_margin,total_quantity
0,West,739813.6085,110798.8170,14.98,12466
1,East,691828.1680,94883.2603,13.71,11159
2,Central,503170.6728,39865.3070,7.92,8820
3,South,391721.9050,46749.4303,11.93,6209


## 5. Data condition checks untuk KPI

In [17]:
checks = {
    'null_order_id': 'SELECT count() FROM silver.silver_orders WHERE order_id IS NULL',
    'null_order_date': 'SELECT count() FROM silver.silver_orders WHERE order_date IS NULL',
    'null_sales': 'SELECT count() FROM silver.silver_orders WHERE sales IS NULL',
    'negative_sales': 'SELECT count() FROM silver.silver_orders WHERE sales < 0',
    'negative_profit': 'SELECT count() FROM silver.silver_orders WHERE profit < 0',
    'duplicate_order_ids': 'SELECT count() - count(distinct order_id) FROM silver.silver_orders',
    'order_after_ship': 'SELECT count() FROM silver.silver_orders WHERE order_date > ship_date',
    'zero_quantity': 'SELECT count() FROM silver.silver_orders WHERE quantity <= 0',
}
for name, query in checks.items():
    result = client.query_df(query)
    print(f'{name}:', int(result.iloc[0, 0]))

null_order_id: 0
null_order_date: 0
null_sales: 0
negative_sales: 0
negative_profit: 1901
duplicate_order_ids: 5083
order_after_ship: 0
zero_quantity: 0
